# Chương 14 — Bài lab: Technical interview

Viết code vào các ô có `# TODO`, rồi chạy ô kiểm tra ngay bên dưới. Ô kiểm tra in **ĐẠT ✔** là xong. Bí quá 20 phút mới mở `loigiai.ipynb`.

Năm lab là **công cụ luyện tập**: chọn chuỗi theo sổ lỗi, lịch ôn, phân tích bản chép lời câu trả lời, kiểm suy dẫn bảng trắng, máy tính chi phí LLM. Sổ lỗi và bản chép lời là **mẫu tự soạn**. Mọi băng thông, dung lượng GPU là **giả định của đề**. Bước cuối mỗi lab (không bắt buộc): thay dữ liệu mẫu bằng dữ liệu của bạn.

In [ ]:
import io, re
import numpy as np
import pandas as pd
np.set_printoptions(precision=4, suppress=True)
GiB = 2**30
# Lab nào có ngẫu nhiên thì tự tạo Generator riêng với seed cố định ngay trong lab đó,
# nên số của lab này không phụ thuộc code bạn viết ở lab khác.

## Lab 1 — Chọn chuỗi theo sổ lỗi
Đọc sổ lỗi (`chuoi,diem,ngay`), tính trọng số luyện lại theo quy tắc của sách, rút 4 chuỗi không lặp cho tuần này. Câu hỏi: vì sao trọng số nhân với số ngày chứ không chỉ theo điểm?

In [ ]:
# Sổ lỗi MẪU (số giả): điểm tự chấm lần gần nhất trên thang 12 của chương 14, và số ngày từ lần luyện đó.
LOG_CSV = '''chuoi,diem,ngay
C01,10,2
C02,6,5
C03,11,1
C04,4,9
C05,9,3
C06,12,0
C07,7,6
C08,5,8
C09,8,4
C10,12,1
C11,3,12
C12,9,2
C13,10,3
C14,6,7
C15,11,1
C16,8,5
C17,7,6
C18,12,0
'''
MAX_SCORE = 12

In [ ]:
def read_log(text_or_path):
    """Đọc sổ lỗi CSV có cột chuoi,diem,ngay. Nhận chuỗi CSV hoặc đường dẫn file. Trả về DataFrame."""
    # TODO
    raise NotImplementedError


def weights(score, days, max_score=MAX_SCORE):
    """Trọng số luyện lại của mỗi chuỗi (quy tắc của sách): (max_score − điểm + 1) · (1 + ngày/7).
    score, days là mảng numpy cùng độ dài. Trả về mảng trọng số."""
    # TODO
    raise NotImplementedError


def pick(log, k=4, seed=0):
    """Rút k chuỗi KHÔNG lặp, xác suất tỉ lệ với trọng số. Dùng np.random.default_rng(seed) và rng.choice
    với replace=False, p=trọng số đã chuẩn hóa. Trả về list tên chuỗi theo thứ tự rút."""
    # TODO
    raise NotImplementedError

In [ ]:
log = read_log(LOG_CSV)
w = weights(log["diem"].to_numpy(), log["ngay"].to_numpy())
p = w / w.sum()
log = log.assign(w=w.round(3), p=p.round(4))
print(log.sort_values("w", ascending=False).head(5).to_string(index=False))
i_max, i_min = int(np.argmax(w)), int(np.argmin(w))
print("lớn nhất:", log.chuoi[i_max], round(w[i_max], 3), "| nhỏ nhất:", log.chuoi[i_min], round(w[i_min], 3),
      "| tỉ số xác suất:", round(p[i_max] / p[i_min], 1))
chosen = pick(log, k=4, seed=0)
print("tuần này luyện (seed 0):", chosen)
# Kiểm chính hàm pick của bạn: 4000 seed khác nhau, mỗi lần rút 1 chuỗi
freq = np.mean([pick(log, k=1, seed=s)[0] == log.chuoi[i_max] for s in range(4000)])
print("tần suất pick rút được", log.chuoi[i_max], "qua 4000 seed:", round(freq, 4), "| xác suất lý thuyết:", round(p[i_max], 4))
assert log.chuoi[i_max] == "C11" and abs(w[i_max] - 27.143) < 1e-3
assert log.chuoi[i_min] == "C06" and abs(w[i_min] - 1.0) < 1e-12
assert len(chosen) == 4 and len(set(chosen)) == 4
assert chosen == pick(log, k=4, seed=0), "cùng seed phải ra cùng lựa chọn"
assert chosen == ["C11", "C05", "C02", "C01"], "seed 0 phải cho đúng lựa chọn đã công bố trong sách"
assert all(len(set(pick(log, k=4, seed=s))) == 4 for s in range(300)), "pick không được rút lặp"
assert abs(freq - p[i_max]) < 0.02, "tần suất rút không khớp xác suất theo trọng số"
print("Lab 1 ĐẠT ✔")

**Bước cuối (không bắt buộc):** `pick(read_log('so_loi.csv'))` với sổ lỗi thật của bạn.

## Lab 2 — Lịch ôn Leitner
Mô phỏng 28 ngày, 18 chuỗi, mỗi ngày thêm một chuỗi mới. Câu hỏi: ngày nặng nhất có bao nhiêu chuỗi, và có vừa một buổi tối không?

In [ ]:
# Hộp Leitner giống trang Ôn tập của app: hộp 1, 2, 3, 4 ôn lại sau 1, 3, 7, 14 ngày; quên thì về hộp 1.
INTERVAL = {1: 1, 2: 3, 3: 7, 4: 14}

def pass_rule(i, d):
    '''Quy tắc qua/trượt XÁC ĐỊNH để số tái lập: chuỗi i ở ngày d trượt khi (7i + d) chia hết cho 5 (qua 80%).'''
    return (i * 7 + d) % 5 != 0

In [ ]:
def schedule(n_items=18, days=28, new_per_day=1, pass_fn=pass_rule, interval=INTERVAL):
    """Mô phỏng lịch ôn. Mỗi ngày d (0..days−1): thêm new_per_day chuỗi mới vào hộp 1, đến hạn ngay ngày d
    (cho tới khi đủ n_items). Mọi chuỗi đến hạn ngày d được luyện; qua thì lên một hộp (tối đa hộp 4), trượt thì về hộp 1;
    hạn mới = d + interval[hộp mới]. Trả về mảng int dài `days`: số chuỗi phải luyện mỗi ngày."""
    # TODO
    raise NotImplementedError

In [ ]:
load = schedule()
print("tải mỗi ngày:", load.tolist())
heavy = [int(d) + 1 for d in np.flatnonzero(load == load.max())]
print("tổng", int(load.sum()), "lượt; ngày nặng nhất", int(load.max()), "chuỗi, vào ngày", heavy,
      "->", int(load.max()) * 12, "phút nói nếu mỗi chuỗi 12 phút")
rng2 = np.random.default_rng(2)                          # RNG riêng: biến thể qua/trượt ngẫu nhiên 80%
load_r = schedule(pass_fn=lambda i, d: rng2.random() < 0.8)
print("biến thể ngẫu nhiên (seed 2):", load_r.tolist(), "| tổng", int(load_r.sum()), "| nặng nhất", int(load_r.max()))
assert load.tolist() == [1, 2, 1, 1, 3, 2, 3, 2, 2, 3, 2, 6, 3, 3, 3, 2, 6, 3, 3, 2, 1, 3, 1, 2, 0, 2, 4, 3]
assert load.sum() == 69 and load.max() == 6 and heavy == [12, 17]
assert schedule(pass_fn=lambda i, d: True)[:4].tolist() == [1, 1, 1, 2]
print("Lab 2 ĐẠT ✔")

## Lab 3 — Phân tích bản chép lời câu trả lời (tuần 38)
Từ, từ/phút và từ đệm đã có sẵn trong `basic_stats` (cùng việc với Lab 4 chương 13). Bạn chỉ viết phần mới: đếm I/we và cụm chữ số. Câu hỏi: công cụ bỏ sót những gì?

In [ ]:
# Bản chép lời MẪU (tự soạn) của một câu trả lời tầng 1 chuỗi C01, nói trong 50 giây.
# Dùng ở tuần 38 cho bản chép lời buổi phỏng vấn thử của bạn.
TRANSCRIPT = ("um so the gradient of cross entropy with respect to the logits is p minus y . "
  "basically you take the softmax output and subtract the one hot label . "
  "uh the reason is that the log of the softmax cancels the exponential , so you know the derivative is linear in p . "
  "we checked it with finite differences and the relative error was about ten to the minus ten . "
  "like I think the key point is that the gradient never saturates when the model is wrong , "
  "so I would say it trains faster than mean squared error with a sigmoid . "
  "um one limitation is that it says nothing about calibration , which I measured separately on 2 000 examples .")
SECONDS = 50
FILLERS = ["um", "uh", "like", "you know", "basically", "kind of", "i think", "i would say"]
# Ngưỡng của SÁCH (không phải của chương trình tuyển nào): 120–160 từ/phút; từ đệm <= 3 trên 100 từ.


def basic_stats(text, seconds, fillers=FILLERS):
    '''ĐƯA SẴN, không phải viết lại: cùng việc với talk_stats ở Lab 4 chương 13 (từ, từ/phút, từ đệm), nhưng nhận
    văn bản và tổng số giây thay cho mốc thời gian từng từ, và khớp cả cụm từ đệm nhiều chữ ("you know").'''
    t = " " + re.sub(r"\s+", " ", text.lower()) + " "
    toks = [w for w in t.split() if re.search(r"[a-z0-9]", w)]
    fill = {f: len(re.findall(r"(?<= )" + re.escape(f) + r"(?= )", t)) for f in fillers}
    n_fill = sum(fill.values())
    return dict(words=len(toks), wpm=round(len(toks) / seconds * 60, 1), fillers=n_fill,
                per100=round(100 * n_fill / len(toks), 1), fill_detail={k: v for k, v in fill.items() if v})

In [ ]:
def voice_and_digits(text):
    """Phần MỚI so với chương 13. Trả về dict: I, we (số token đúng bằng 'i', 'we' sau khi chuyển chữ thường,
    token tách theo khoảng trắng), digits (số cụm chữ số trong văn bản gốc; '2 000' tính là một cụm)."""
    # TODO
    raise NotImplementedError

In [ ]:
r = {**basic_stats(TRANSCRIPT, SECONDS), **voice_and_digits(TRANSCRIPT)}
print(r)
print("tốc độ trong vùng 120–160:", 120 <= r["wpm"] <= 160, "| từ đệm <= 3 trên 100 từ:", r["per100"] <= 3)
clean = re.sub(r"\b(um|uh|basically|you know|like|i think|i would say)\b ?", "", TRANSCRIPT.lower())
print("bỏ hết từ đệm:", basic_stats(clean, SECONDS)["words"], "từ")
assert r["words"] == 116 and abs(r["wpm"] - 139.2) < 0.05 and r["fillers"] == 8
assert r["I"] == 3 and r["we"] == 1 and r["digits"] == 1
assert voice_and_digits("i said it is fine, We ran 3 seeds on 2 000 items and I think we won") == dict(I=2, we=2, digits=2)
print("Lab 3 ĐẠT ✔")

## Lab 4 — Kiểm suy dẫn bảng trắng
Bước 1: gradient số bằng sai phân trung tâm (float64), so với gradient khẳng định. Bước 2: hai trong bốn khẳng định sai; tìm ra khẳng định nào, suy dẫn lại trên giấy và điền bản sửa vào `FIXES`.

In [ ]:
rng4 = np.random.default_rng(1)          # RNG riêng của lab 4
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()
def layernorm(v):                         # LayerNorm không affine, phương sai chia n (không chia n − 1)
    mu = v.mean(); s = np.sqrt(((v - mu) ** 2).mean())
    return (v - mu) / s

z = rng4.normal(size=5); y = np.eye(5)[1]
A = rng4.normal(size=(4, 4)); x = rng4.normal(size=4)
X = rng4.normal(size=(6, 3)); W = rng4.normal(size=(3, 2)); Y = rng4.normal(size=(6, 2))
v = rng4.normal(size=6); u = rng4.normal(size=6)                 # u: vector chiếu, kiểm Jacobian qua hàm vô hướng uᵀLN(v)
n = 6                                                            # σ của LN(v) là v.std() (chia n)

# Bốn khẳng định "viết trên bảng". Mỗi khẳng định: (hàm vô hướng f, điểm x0, gradient khẳng định tại x0)
CLAIMS = {
  "a) dCE/dz = p − y":                   (lambda z_: -np.log(softmax(z_)[1]), z, softmax(z) - y),
  "b) d(xᵀAx)/dx = 2Ax":                 (lambda x_: x_ @ A @ x_, x, 2 * A @ x),
  "c) d‖XW − Y‖²/dW = 2Xᵀ(XW − Y)":       (lambda W_: np.sum((X @ W_ - Y) ** 2), W, 2 * X.T @ (X @ W - Y)),
  "d) d(uᵀLN(v))/dv = (I − 11ᵀ/n)u/σ":   (lambda v_: u @ layernorm(v_), v, (np.eye(n) - np.ones((n, n)) / n) @ u / v.std()),
}

In [ ]:
def num_grad(f, x0, h=1e-6):
    """Gradient số bằng sai phân trung tâm (f(x + h·eᵢ) − f(x − h·eᵢ))/(2h), cho x0 có shape bất kỳ (float64)."""
    # TODO
    raise NotImplementedError


def rel_err(a, b):
    """‖a − b‖ / max(‖a‖ + ‖b‖, 1e-12)."""
    # TODO
    raise NotImplementedError


def check(f, x0, g_claim, tol=1e-6):
    """Trả về (sai số tương đối giữa gradient số và gradient khẳng định, 'ĐÚNG' nếu < tol, ngược lại 'SAI')."""
    # TODO
    raise NotImplementedError

In [ ]:
# Bước 1: phán quyết của check() cho bốn khẳng định
res = {name: check(f, x0, g) for name, (f, x0, g) in CLAIMS.items()}
for name, (e, verdict) in res.items():
    print(f"{name:40s} rel err {e:.2e} -> {verdict}")

In [ ]:
# Bước 2: với MỖI khẳng định mà ô trên báo SAI,
# suy dẫn lại trên giấy và viết gradient đúng tại điểm x0 của khẳng định đó.
# Khóa của FIXES: tên khẳng định, chép đúng như trong CLAIMS. Giá trị: mảng cùng shape với x0, tính bằng CÔNG THỨC
# từ các biến có sẵn (z, y, A, x, X, W, Y, v, u, n, softmax, layernorm). Không dùng num_grad ở đây:
# điền gradient số là chép đáp án, không phải sửa suy dẫn.
FIXES = {}
# TODO
raise NotImplementedError

In [ ]:
res = {name: check(f, x0, g) for name, (f, x0, g) in CLAIMS.items()}
# (1) num_grad và rel_err đúng trên hàm đã biết đạo hàm: f(t) = Σ t·sin t, f'(t) = sin t + t·cos t
ts = np.linspace(-1.0, 2.0, 7)
assert rel_err(num_grad(lambda t: np.sum(t * np.sin(t)), ts), np.sin(ts) + ts * np.cos(ts)) < 1e-8, "num_grad chưa đủ chính xác"
assert all(np.isclose(rel_err(c * ts, c * (ts + 0.1)), rel_err(ts, ts + 0.1)) for c in (1e-6, 1e6)), "rel_err phải không đổi khi nhân cả hai vế cùng hệ số"
# (2) phán quyết và bản sửa của bạn
wrong = [k for k, (e, verdict) in res.items() if verdict == "SAI"]
assert len(wrong) == 2, "đề nói đúng hai khẳng định sai"
assert all(res[k][0] < 1e-8 for k in CLAIMS if k not in wrong), "khẳng định đúng phải khớp tới cỡ 1e-10"
assert set(FIXES) == set(wrong), "khóa của FIXES phải đúng là các khẳng định bị báo SAI"
for k in wrong:
    f, x0, g_claim = CLAIMS[k]
    e, verdict = check(f, x0, FIXES[k])
    print(f"bản sửa của {k[:2]} rel err {e:.2e} -> {verdict}")
    assert verdict == "ĐÚNG" and rel_err(FIXES[k], g_claim) > 1e-3
print("Lab 4 ĐẠT ✔")

## Lab 5 — Máy tính chi phí LLM
Trọng số, KV cache, FLOPs huấn luyện theo quy ước chương 8, cận dưới ms/token. Mọi băng thông và dung lượng GPU là giả định.

In [ ]:
# Cấu hình hình dạng Llama-2-7B (như chương 8). n_kv = số đầu K/V; n_kv = n_h là multi-head attention (MHA).
CFG = dict(L=32, d=4096, n_h=32, n_kv=32, V=32000, ffn=11008)
CFG_KV8 = dict(CFG, n_kv=8)            # GIẢ ĐỊNH để so sánh: 8 đầu K/V (không phải Llama-2-7B)
BW = 2e12                              # băng thông GIẢ ĐỊNH 2 TB/s (byte/giây)
Q4 = 0.5 + 2 / 64                      # 4-bit cộng một scale fp16 cho mỗi khối 64 trọng số: byte/trọng số

def count_params(c):
    '''Số tham số (đã kiểm ở chương 8): attention, MLP SwiGLU 3 ma trận, 2 RMSNorm mỗi lớp,
    embedding vào và lm_head KHÔNG buộc trọng số, một RMSNorm cuối.'''
    dh = c["d"] // c["n_h"]
    att = 2 * c["d"] * c["d"] + 2 * c["d"] * c["n_kv"] * dh
    return c["L"] * (att + 3 * c["d"] * c["ffn"] + 2 * c["d"]) + 2 * c["V"] * c["d"] + c["d"]

In [ ]:
def weight_gib(c, nbytes=2):
    """Bộ nhớ trọng số, đơn vị GiB (2³⁰ byte)."""
    # TODO
    raise NotImplementedError


def kv_bytes(c, T, B=1, nbytes=2):
    """Số byte KV cache cho B chuỗi, mỗi chuỗi T token. Tự suy công thức từ hình dạng các tensor được lưu ở mỗi lớp."""
    # TODO
    raise NotImplementedError


def train_flops(c, D):
    """FLOPs huấn luyện ≈ 6·N·D theo QUY ƯỚC CHƯƠNG 8: N là tổng tham số trừ bảng embedding đầu vào (V·d),
    vì tra bảng không phải nhân ma trận (N vẫn gồm các RMSNorm, tuy nhỏ). Không tính số hạng attention theo T."""
    # TODO
    raise NotImplementedError


def decode_ms(c, bw=BW, nbytes=2, T=0):
    """Cận dưới ms/token khi giải mã batch 1, bị chặn bởi băng thông: đọc toàn bộ trọng số
    cộng KV cache của T token đã có, chia cho bw."""
    # TODO
    raise NotImplementedError

In [ ]:
N = count_params(CFG)
rows = [("tham số", N), ("trọng số fp16, GiB", round(weight_gib(CFG), 2)), ("trọng số 4-bit + scale, GiB", round(weight_gib(CFG, Q4), 2)),
        ("KV mỗi token, byte", kv_bytes(CFG, 1)), ("KV 4096 token MHA, GiB", round(kv_bytes(CFG, 4096) / GiB, 2)),
        ("KV 4096 token, 8 đầu K/V, GiB", round(kv_bytes(CFG_KV8, 4096) / GiB, 2)),
        ("FLOPs 6·N·D, D = 2·10¹²", f"{train_flops(CFG, 2e12):.3e}"),
        ("decode ms/token, 2 TB/s, chỉ trọng số", round(decode_ms(CFG), 2)), ("decode ms/token, cộng KV 4096", round(decode_ms(CFG, T=4096), 2))]
for k_, v_ in rows:
    print(f"{k_:40s} {v_}")
free = 80 - weight_gib(CFG)
print("GPU giả định 80 GiB, còn", round(free, 2), "GiB -> MHA", int(free // (kv_bytes(CFG, 4096) / GiB)),
      "chuỗi 4096 token; 8 đầu K/V", int(free // (kv_bytes(CFG_KV8, 4096) / GiB)), "(chưa trừ activation, phân mảnh)")
lap = weight_gib(CFG, Q4) + kv_bytes(CFG, 4096) / GiB
print("laptop 6 GiB: 4-bit + KV 4096 =", round(lap, 2), "GiB; KV 2048:", round(weight_gib(CFG, Q4) + kv_bytes(CFG, 2048) / GiB, 2), "GiB (chưa tính overhead)")
print("nếu KHÔNG bỏ embedding đầu vào: 6ND =", f"{6 * N * 2e12:.3e}", "-> phải nói mình dùng quy ước nào")
assert N == 6738415616
assert abs(weight_gib(CFG) - 12.551) < 1e-3 and abs(weight_gib(CFG, Q4) - 3.334) < 1e-3
assert kv_bytes(CFG, 1) == 524288 and kv_bytes(CFG_KV8, 4096) * 4 == kv_bytes(CFG, 4096)
assert kv_bytes(CFG, 4096, B=8) == 16 * GiB, "KV cache phải tỉ lệ với batch (ví dụ 14.6: batch 8 là 16 GiB)"
assert abs(train_flops(CFG, 2e12) - 7.929e22) / 7.929e22 < 1e-3
assert abs(decode_ms(CFG) - 6.738) < 1e-3 and abs(decode_ms(CFG, T=4096) - 7.812) < 1e-3
print("Lab 5 ĐẠT ✔")

## Săn bug — báo cáo chi phí "trông hợp lý"
Hàm `cost_report` dưới đây chạy không báo lỗi. Triệu chứng: với cấu hình 8 đầu K/V, KV cache 4096 token **bằng đúng** cấu hình MHA; số GiB của trọng số lớn hơn số chương 8 khoảng 7%; FLOPs huấn luyện nhỏ hơn phép tính tay của bạn. Có **bốn** lỗi. Sửa tại chỗ rồi chạy ô kiểm tra.

In [ ]:
# Báo cáo chi phí phục vụ và huấn luyện do một bạn cùng nhóm viết. Chạy không báo lỗi, số trông "hợp lý".
# Sửa tại chỗ trong hàm cost_report, rồi chạy ô kiểm tra bên dưới.
def cost_report(c, T=4096, B=1, nbytes=2, D=2e12, bw=BW, gpu_gib=80):
    n = count_params(c)
    dh = c["d"] // c["n_h"]
    weights = n * nbytes / 1e9
    kv_tok = c["L"] * c["n_h"] * dh * nbytes
    kv_seq = kv_tok * T * B / GiB
    n_mm = n - c["V"] * c["d"]
    return dict(weights_gib=round(weights, 2),
                kv_gib=round(kv_seq, 2),
                max_seqs=int((gpu_gib - weights) // kv_seq),
                train_flops=2 * n_mm * D,
                decode_ms=round(n * nbytes / bw * 1e3, 2))

In [ ]:
def failing_checks(report_fn):
    '''Trả về số phép kiểm chưa đạt (so với các số chuẩn đã có trong sách).'''
    r1, r8, rb = report_fn(CFG), report_fn(CFG_KV8), report_fn(CFG, B=8)
    expect = [(r1["weights_gib"], 12.55), (r1["kv_gib"], 2.00), (r8["kv_gib"], 0.50), (rb["kv_gib"], 16.00),
              (r1["max_seqs"], 33), (r1["train_flops"], 7.929e22), (r1["decode_ms"], 6.74)]
    return sum(abs(got - want) > 0.006 * abs(want) for got, want in expect)

r = cost_report(CFG)
print("cấu hình MHA     :", r)
print("cấu hình 8 đầu K/V:", cost_report(CFG_KV8))
bad = failing_checks(cost_report)
print("Số phép kiểm chưa đạt:", bad, "/ 7")
assert bad == 0, "Còn phép kiểm chưa đạt."
print("Săn bug ĐẠT ✔")